# Сделано "Практика: Word2Vec и простой retrieval" Зейналовым У.Р.

В этом ноутбуке 5 задач:
1. Берём готовые эмбеддинги Word2Vec/аналог (через `gensim`) и смотрим ближайших соседей для слов из корпуса **20 Newsgroups**.
2. Обучаем свой **Skip-gram Word2Vec** на подкорпусе про **хоккей** и смотрим ближайшие слова к названиям команд.
3. Строим **индекс документов**: представляем документ как среднее Word2Vec (опционально с TF–IDF весами) и ищем ближайший документ по запросу.
4. Делаем то же, но через **FAISS** (быстрый nearest-neighbor поиск).
5. (Опционально) Сравниваем cosine vs L2 и обсуждаем, что меняется при нормировке.

> Подсказка: если что-то не скачивается (нет интернета), используйте вариант обучения своих эмбеддингов на корпусе из пункта 2 и применяйте их в пунктах 1/3/4.


## 0) Установка и импорты

Если `gensim` или `faiss` не установлены в вашей среде, раскомментируйте `pip install` ниже.

In [1]:
!pip -q install gensim
!pip -q install faiss-cpu  # для CPU-индекса (Windows иногда требует перезапуск kernel)

import re
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec


## 1) Данные: 20 Newsgroups

Загрузим корпус и подготовим простую токенизацию.

In [16]:
from datasets import load_dataset
import re

ds = load_dataset("data-silence/rus_news_classifier")

data = ds["train"]

texts = data["news"]          # тексты
targets = data["labels"]       # метки
target_names = {
 'climate': 0,
 'conflicts': 1,
 'culture': 2,
 'economy': 3,
 'gloss': 4,
 'health': 5,
 'politics': 6,
 'science': 7,
 'society': 8,
 'sports': 9,
 'travel': 10
}  # названия классов

print("Документов:", len(texts))
print("Категорий:", len(target_names))
print("Категории:", target_names)
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[а-яё0-9]+", text)
    return tokens
tokenized_texts = [tokenize(t) for t in texts]

print("Пример текста:", texts[3][:200])
print("Пример токенов:", tokenized_texts[3][:30])


Документов: 57530
Категорий: 11
Категории: {'climate': 0, 'conflicts': 1, 'culture': 2, 'economy': 3, 'gloss': 4, 'health': 5, 'politics': 6, 'science': 7, 'society': 8, 'sports': 9, 'travel': 10}
Пример текста: Британский боксер-тяжеловес Тайсон Фьюри оскорбил украинца Александра Усика. Его слова приводит Daily Mail. Спортсмен назвал возможного соперника маленьким украинским бомжом и заявил, что ему не интер
Пример токенов: ['британский', 'боксер', 'тяжеловес', 'тайсон', 'фьюри', 'оскорбил', 'украинца', 'александра', 'усика', 'его', 'слова', 'приводит', 'спортсмен', 'назвал', 'возможного', 'соперника', 'маленьким', 'украинским', 'бомжом', 'и', 'заявил', 'что', 'ему', 'не', 'интересен', 'бой', 'с', 'ним', 'это', 'единственное']


## 2) Задание 1: готовые эмбеддинги и ближайшие соседи

Попробуем загрузить готовые вектора через `gensim.downloader`.

Рекомендуемые варианты:
- `glove-wiki-gigaword-100` (обычно быстрее скачивается)
- `word2vec-google-news-300` (очень большой)

Если скачивание недоступно — пропустите к заданию 2 (обучение своих эмбеддингов) и используйте их вместо готовых.

In [15]:
#Посмотреть, какие модели доступны:
print(list(api.info()['models'].keys())[:30])

MODEL_NAME = "word2vec-ruscorpora-300"  # можно поменять

try:
    wv = api.load(MODEL_NAME)  # KeyedVectors
    print("Загружено:", MODEL_NAME)
    print("Размерность:", wv.vector_size)
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))
    print("Продолжайте с обучением собственных эмбеддингов ниже (Задание 2).")


['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']
Загружено: word2vec-ruscorpora-300
Размерность: 300


### 2.1 Выбираем слова из корпуса и смотрим соседей

Выберите 10–20 слов (желательно тематических) и посмотрите ближайшие соседи.

**Задача:**
- Выберите по 3–5 слов из разных доменов (например, религия/спорт/компьютеры/медицина).
- Для каждого слова распечатайте 10 ближайших соседей.
- Сделайте 3–5 наблюдений: где соседи хорошие, где странные, почему так могло получиться.

### 2.3 Семантические сдвиги (vector offsets)

Идея: отношения между словами часто кодируются **разностями векторов**.

**Что проверить:**
- Сдвиг `queen - king` похож на `woman - man`
- Сдвиг `paris - france` похож на `berlin - germany` (и т.п.)

Ниже мы:
1) найдём ближайшие слова к вектору-сдвигу (например, к `queen-king`),
2) посчитаем косинусное сходство между двумя сдвигами.

**Пояснение выполняющего:**

В модели word2vec-ruscorpora-300 есть улсовность что слова не пишуться без его обозначение, то есть если на нужно найти "король" надо писать "король_NOUN"



In [19]:
def pick_embeddings_for_offsets():
    # Приоритет: готовые wv -> fastText ft_wv -> None
    if 'wv' in globals() and wv is not None:
        return wv, 'pretrained (wv)'

emb, emb_name = pick_embeddings_for_offsets()
print('Embeddings for offsets:', emb_name)

def unit(v):
    v = v.astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-9)

def nearest_to_vector(emb, vec, topn=10, ban=()):
    # gensim KeyedVectors умеют most_similar по вектору
    res = emb.most_similar([vec], topn=topn + len(ban))
    out = []
    for w, s in res:
        if w not in ban:
            out.append((w, s))
        if len(out) >= topn:
            break
    return out

def offset(word_a, word_b):
    # vec(word_a) - vec(word_b)
    return emb.get_vector(word_a) - emb.get_vector(word_b)

def try_offset_demo(pair1, pair2, topn=10):
    (a1, b1) = pair1
    (a2, b2) = pair2
    missing = [w for w in [a1,b1,a2,b2] if w not in emb]
    if missing:
        print('Пропускаю, нет слов в словаре:', missing)
        return
    d1 = offset(a1, b1)
    d2 = offset(a2, b2)
    sim = float(unit(d1) @ unit(d2))
    print(f"offset({a1}-{b1}) vs offset({a2}-{b2}) cosine = {sim:.3f}")
    print(f"Nearest to vector ({a1}-{b1}):", [w for w,_ in nearest_to_vector(emb, d1, topn=topn, ban=(a1,b1))])
    print(f"Nearest to vector ({a2}-{b2}):", [w for w,_ in nearest_to_vector(emb, d2, topn=topn, ban=(a2,b2))])
    print()

if emb is None:
    print('Нет подходящих эмбеддингов. Сначала загрузите wv или ft_wv.')
else:
    # 1) queen-king ~ woman-man
    try_offset_demo(('король_NOUN','королева_NOUN'), ('мужчина_NOUN','женщина_NOUN'), topn=10)

    # 2) paris-france ~ berlin-germany (и ещё несколько примеров)
    try_offset_demo(('париж_NOUN','франция_NOUN'), ('берлин_NOUN','германия_NOUN'), topn=10)
    try_offset_demo(('рим_NOUN','италия_NOUN'), ('мадрид_NOUN','испания_NOUN'), topn=10)

    # 3) Классическая аналогия: king - man + woman ≈ queen
    needed = ['король_NOUN','мужчина_NOUN','женщина_NOUN']
    if all(w in emb for w in needed):
        v = emb.get_vector('король_NOUN') - emb.get_vector('мужчина_NOUN') + emb.get_vector('женщина_NOUN')
        print("Analogy king - мужчина_NOUN + женщина_NOUN ->", nearest_to_vector(emb, v, topn=10, ban=('король_NOUN','мужчина_NOUN','женщина_NOUN'))[:10])
    else:
        print('Пропускаю analogy king-man+woman: не хватает слов')


Embeddings for offsets: pretrained (wv)
offset(король_NOUN-королева_NOUN) vs offset(мужчина_NOUN-женщина_NOUN) cosine = 0.254
Nearest to vector (король_NOUN-королева_NOUN): ['семиградский_ADJ', 'пилсудский_ADJ', 'цесарь_NOUN', 'гетман::жолкевский_NOUN', 'седмиградский_ADJ', 'турский::салтан_NOUN', 'баторий_NOUN', 'господарь::молдавский_NOUN', 'отчина_NOUN', 'золотаренок_NOUN']
Nearest to vector (мужчина_NOUN-женщина_NOUN): ['усищи_NOUN', 'взрослый_ADJ', 'минивэн_NOUN', 'мужской_ADJ', 'гладко::выбривать_VERB', 'половозрелый_ADJ', 'лысеть_VERB', 'бриться_VERB', 'конкурсант_NOUN', 'тестостерон_NOUN']

offset(париж_NOUN-франция_NOUN) vs offset(берлин_NOUN-германия_NOUN) cosine = 0.785
Nearest to vector (париж_NOUN-франция_NOUN): ['нью-йорк_NOUN', 'passy_NOUN', 'монпарнасский_ADJ', 'курфюрстендамм_NOUN', 'петербург_NOUN', 'чудинка_NOUN', 'кабаре_NOUN', 'озимовский_ADJ', 'берлин_NOUN', 'ша::нуар_INTJ']
Nearest to vector (берлин_NOUN-германия_NOUN): ['курфюрстендамм_NOUN', 'вена_NOUN', 'филер

In [20]:
def show_neighbors(wv, word, topn=10):
    if wv is None:
        print("Нет загруженных эмбеддингов (wv=None).")
        return
    if word not in wv:
        print(f"'{word}' нет в словаре модели.")
        return
    neigh = wv.most_similar(word, topn=topn)
    print(word, "->", [w for w, _ in neigh])

words_to_try = ["хокей_NOUN", "окно_NOUN", "исус_NOUN", "доктор_NOUN", "машина_NOUN", "космос_NOUN", "компьютер_NOUN"]

for w in words_to_try:
    show_neighbors(wv, w, topn=10)


'хокей_NOUN' нет в словаре модели.
окно_NOUN -> ['окошко_NOUN', 'оконце_NOUN', 'форточка_NOUN', 'ставня_NOUN', 'подоконник_NOUN', 'занавеска_NOUN', 'оконный_ADJ', 'незавешивать_VERB', 'дверь_NOUN', 'балкон_NOUN']
исус_NOUN -> ['егда::приидеши_NOUN', 'христос_NOUN', 'иисус_NOUN', 'пресвят_NOUN', 'свидение_NOUN', 'преосвященный::владыко_NOUN', 'нечестивия_NOUN', 'господи_NOUN', 'услыш_NOUN', 'внидет_ADV']
доктор_NOUN -> ['гаспар::арнерить_VERB', 'профессор_NOUN', 'врач_NOUN', 'гонориса::кауз_NOUN', 'штофреген_NOUN', 'несмеянова_NOUN', 'варвинский_ADJ', 'матцк_NOUN', 'лекарь_NOUN', 'кроссет_NOUN']
машина_NOUN -> ['автомобиль_NOUN', 'грузовик_NOUN', 'джип_NOUN', 'мерседес_NOUN', 'легковой_ADJ', 'жигули_NOUN', 'трактор_NOUN', 'мотоцикл_NOUN', 'мотор_NOUN', 'автомашина_NOUN']
космос_NOUN -> ['космический_ADJ', 'космонавт_NOUN', 'межпланетный_ADJ', 'околоземный_ADJ', 'околосолнечный_ADJ', 'орбитальный_ADJ', 'планета_NOUN', 'марсоход_NOUN', 'пилотировать_VERB', 'околоземный::орбита_NOUN']
комп

## 3) Задание 2: обучаем Word2Vec в категории политика

Возьмём 9 тему в базе данных `политика` и обучим **Skip-gram**.

**Задача:**
- Обучите модель.
- Проверьте ближайшие слова к названиям команд: `Путин`, `Трамп`, `Россия`, `Китай` и т.п.
- Сравните с готовыми эмбеддингами (если они доступны).

In [47]:
# Вытащим только хоккей
# берём только категорию "sports"
sports_data = [x for x in data if x["labels"] == 6]

sports_texts = [x["news"] for x in sports_data]
sports_tok = [tokenize(t) for t in sports_texts]

print("Политические документов:", len(sports_tok))
print("Пример токенов:", sports_tok[0][:30])


Политические документов: 5558
Пример токенов: ['планы', 'эстонии', 'по', 'обучению', 'людей', 'партизанской', 'войне', 'говорят', 'о', 'том', 'что', 'в', 'минобороны', 'страны', 'нет', 'психиатра', 'об', 'этом', 'написал', 'спикер', 'госдумы', 'вячеслав', 'володин', 'в', 'своем', 'канале', '5', 'сентября', 'вице', 'канцлер']


In [45]:
# Обучение skip-gram: sg=1
# Параметры можно менять
w2v_hockey = Word2Vec(
    sentences=sports_texts,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,
    epochs=10,
    workers=4
)

wv_hockey = w2v_hockey.wv
print("Словарь политической модели:", len(wv_hockey))
print("Размерность:", wv_hockey.vector_size)
print(wv_hockey)


Словарь политической модели: 169
Размерность: 100
KeyedVectors<vector_size=100, 169 keys>


In [46]:
teams = ["путин", "трамп", "россия", "китай", "америка", "мадура"]
for t in teams:
    if t in wv_hockey:
        print("\n", t, "->", [w for w, _ in wv_hockey.most_similar(t, topn=10)])
    else:
        print("\n", t, "нет в словаре (проверь min_count или орфографию)")



 путин нет в словаре (проверь min_count или орфографию)

 трамп нет в словаре (проверь min_count или орфографию)

 россия нет в словаре (проверь min_count или орфографию)

 китай нет в словаре (проверь min_count или орфографию)

 америка нет в словаре (проверь min_count или орфографию)

 мадура нет в словаре (проверь min_count или орфографию)


## 4) Задание 3: индекс документов как матрица (mean Word2Vec)

Построим эмбеддинг документа как **среднее** эмбеддингов слов.

Два варианта:
1. Просто среднее.
2. **TF–IDF взвешенное** среднее (обычно лучше).

**Задача:**
- Постройте матрицу `D` размера (num_docs × dim).
- Напишите 3–5 запросов и посмотрите топ-5 ближайших документов.
- Оцените глазами: насколько выдача “в тему”.

In [55]:
# texts — колонка 'content' из Dataset
texts_list = [t for t in texts]   # <- конвертируем в обычный список

# делим данные
X_train, X_test = train_test_split(texts_list, test_size=0.2, random_state=42)

docs = X_test[:1000]
docs_tok = [tokenize(t) for t in docs]

# используем твою обученную модель
wv_use = wv_hockey
print("Используем эмбеддинги: wv_hockey")

DIM = wv_use.vector_size

def doc_vector_mean(tokens, wv_use):
    vecs = [wv_use[w] for w in tokens if w in wv_use]
    if not vecs:
        return np.zeros(DIM, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# TF-IDF
vectorizer = TfidfVectorizer(tokenizer=tokenize, lowercase=True, min_df=2, max_df=0.9)
tfidf = vectorizer.fit_transform(docs)
vocab = vectorizer.vocabulary_

def doc_vector_tfidf(tokens, wv_use, tfidf_row, vocab):
    weights = {}
    for w in tokens:
        j = vocab.get(w, None)
        if j is not None:
            weights[w] = tfidf_row[0, j]

    num = np.zeros(DIM, dtype=np.float32)
    den = 0.0

    for w, a in weights.items():
        if w in wv_use and a > 0:
            num += (a * wv_use[w]).astype(np.float32)
            den += float(a)

    if den == 0.0:
        return doc_vector_mean(tokens, wv_use)

    return (num / den).astype(np.float32)

# считаем матрицы
D_mean = np.vstack([doc_vector_mean(t, wv_use) for t in docs_tok])
D_tfidf = np.vstack([doc_vector_tfidf(docs_tok[i], wv_use, tfidf[i], vocab) for i in range(len(docs_tok))])

print("D_mean shape:", D_mean.shape)
print("D_tfidf shape:", D_tfidf.shape)

# нормализация
D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)

print('D_mean_n shape:', D_mean_n.shape)
print('D_tfidf_n shape:', D_tfidf_n.shape)

Используем эмбеддинги: wv_hockey


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


D_mean shape: (1000, 100)
D_tfidf shape: (1000, 100)
D_mean_n shape: (1000, 100)
D_tfidf_n shape: (1000, 100)


In [56]:
def normalize_rows(M: np.ndarray) -> np.ndarray:
    M = M.astype(np.float32, copy=False)
    norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M / norms

def cosine_topk_pre_norm(query_vec: np.ndarray, Dnrm: np.ndarray, k: int = 5):
    """Cosine top-k, where Dnrm already has unit-norm rows."""
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]


In [58]:
# 5 запросов + вызовы: TF–IDF baseline + dense cosine

import re
import numpy as np

queries = [
    "ошибка установки драйвера виндовс",
    "футбольная команда победила",
    "компания наса отправила в космос",
    "компьютерная графика рендерит полигон",
    "доктор узнал симптомы болезни",
]
K = 5

for q in queries:
    print("=" * 90)
    print("Query:", q)

    # (1) TF–IDF baseline: scores = tfidf @ q_tfidf^T
    q_tfidf = vectorizer.transform([q])                    # (1, |V|)
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()   # (N,)
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\nTF–IDF top-k:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    # (2) Dense cosine over pre-normalized doc matrix D_tfidf_n
    tok = tokenize(q)
    qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab)     # (dim,)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\nDense (cosine) top-k:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")

Query: ошибка установки драйвера виндовс

TF–IDF top-k:
 1. score=0.2204 | Пользователи актуальной ОС от Microsoft рассказали о проблеме со встроенной программой «Проводник». Об этом сообщает издание TechSpot. По словам журналистов портала, посетители Reddit обратили внимани...
 2. score=0.1880 | Запуск марсохода Perseverance ("Настойчивость") перенесли с 20 на 22 июля. Как сообщили в NASA, перенос связан с непредвиденной задержкой, возникшей во время установки марсохода под обтекатель ракеты-...
 3. score=0.1201 | Очередное обновление для Windows 11 вызвало проблемы в работе операционной системы. Об этом сообщает издание BleepingComputer. Журналисты медиа обратили внимание на жалобы владельцев ПК под управление...
 4. score=0.0801 | Российскую тяжелую огнеметную систему (ТОС) следует обнаружить и обезвредить до ее применения. Об этом пишет Newsweek. Американское издание приводит комментарий оборонного эксперта Дэвида Хэмблинга, к...
 5. score=0.0772 | Первые обзорщики очков дополненно

### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).
